# AI-Powered Travel Agent

Generate personalized travel itineraries using OpenRouter. Customize your trip and export to calendar.


In [1]:
# Install dependencies
!pip install requests icalendar

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Cell 1: Imports & Setup
import os
import re
import requests
from IPython.display import display, Markdown
from icalendar import Calendar, Event
from datetime import datetime, date, timedelta
import uuid
import json

print("✓ All imports successful")

✓ All imports successful


/Users/prathamnawal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
# Cell 2: API Key Configuration
# Replace with your OpenRouter API key from https://openrouter.ai
OPENROUTER_API_KEY = "your open router api key here"

if OPENROUTER_API_KEY == "your open router api key here":
    print("⚠️  Please set your OpenRouter API key above")
else:
    print("✓ API key configured")

⚠️  Please set your OpenRouter API key above


In [5]:
# Cell 3: Trip Inputs - Edit these values
destination = "New Delhi, India"
num_days = 5
budget = "luxury"                              # "low" | "mid" | "luxury"
interests = ["food", "culture", "history"]  # e.g., ["adventure", "nature", "nightlife"]
companions = "couple"                       # "solo" | "couple" | "family" | "group"
trip_start_date = "2026-04-07"              # YYYY-MM-DD format

print(f"Trip Summary:")
print(f"  Destination: {destination}")
print(f"  Duration: {num_days} days")
print(f"  Budget: {budget}")
print(f"  Interests: {', '.join(interests)}")
print(f"  Companions: {companions}")
print(f"  Start Date: {trip_start_date}")

Trip Summary:
  Destination: New Delhi, India
  Duration: 5 days
  Budget: luxury
  Interests: food, culture, history
  Companions: couple
  Start Date: 2026-04-07


In [7]:
# Cell 4: Core Agent Function
def generate_itinerary(destination, num_days, budget, interests, companions):
    """
    Generate a personalized travel itinerary using OpenRouter.
    
    Returns:
        str: Structured itinerary with days and activities
    """
    
    # Build the system prompt
    system_prompt = """You are an expert travel agent specializing in creating personalized travel itineraries.
Your task is to create a detailed, day-by-day travel plan.

IMPORTANT: Format your response EXACTLY like this:

Day 1 - [City/Area Name]
- Activity 1
- Activity 2
- Activity 3

Day 2 - [City/Area Name]
- Activity 1
- Activity 2

... and so on

Each activity should be a specific, actionable recommendation. Include restaurant names, landmark names, museums, etc.
Make sure activities match the budget tier and traveler interests."""
    
    # Build the user prompt
    user_prompt = f"""Create a {num_days}-day travel itinerary for:
- Destination: {destination}
- Budget Level: {budget}
- Interests: {', '.join(interests)}
- Traveling with: {companions}

Please include a mix of iconic attractions, hidden gems, dining recommendations, and local experiences.
Make sure the itinerary is realistic and accounts for travel time between locations."""
    
    # Call OpenRouter API
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "openai/gpt-4o-mini",  # Fast and capable model
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 2000
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        itinerary = result["choices"][0]["message"]["content"]
        return itinerary
    
    except requests.exceptions.RequestException as e:
        return f"Error calling OpenRouter API: {str(e)}\n\nMake sure your API key is valid."

print("✓ generate_itinerary() function ready")

✓ generate_itinerary() function ready


In [8]:
# Cell 5: Generate and Display Itinerary
print("Generating itinerary... (this may take a moment)\n")

itinerary = generate_itinerary(destination, num_days, budget, interests, companions)

# Display as rich markdown
markdown_output = f"""## {destination} - {num_days} Day Itinerary

**Trip Details:**
- Budget: {budget.capitalize()}
- Interests: {', '.join(interests)}
- Companions: {companions.capitalize()}

---

{itinerary}
"""

display(Markdown(markdown_output))

Generating itinerary... (this may take a moment)



## New Delhi, India - 5 Day Itinerary

**Trip Details:**
- Budget: Luxury
- Interests: food, culture, history
- Companions: Couple

---

Day 1 - New Delhi
- Arrive at Indira Gandhi International Airport and check into The Oberoi, New Delhi, a luxury hotel known for its exquisite hospitality.
- Visit the India Gate, an iconic war memorial, and take a leisurely walk around the lush green lawns.
- Enjoy a fine dining experience at Bukhara, renowned for its North Indian cuisine, particularly the dal Bukhara and kebabs.

Day 2 - New Delhi
- Start your day with a visit to the beautiful Humayun's Tomb, a UNESCO World Heritage Site showcasing Mughal architecture.
- Explore the vibrant lanes of Chandni Chowk, sampling street food delicacies like parathas at Paranthe Wali Gali, and visit the historic Jama Masjid.
- In the evening, indulge in a romantic dinner at Indian Accent, offering modern Indian cuisine in an elegant setting.

Day 3 - New Delhi
- Experience a guided tour of the Qutub Minar, another UNESCO World Heritage Site, and admire the intricate carvings and towering minaret.
- Head to the National Museum to delve into India's rich history and culture, exploring its vast collection of artifacts.
- Enjoy a luxurious dinner at the Spice Route, known for its Southeast Asian cuisine and beautiful ambiance.

Day 4 - New Delhi
- Visit the Lotus Temple, a stunning architectural marvel, and spend some quiet time meditating in its tranquil surroundings.
- Take a private guided tour of the National Gallery of Modern Art to explore contemporary Indian art.
- Savor a sumptuous dinner at The Pind Balluchi, which offers an authentic Punjabi dining experience in a rustic setting.

Day 5 - New Delhi
- Spend the morning at the Red Fort, soaking in the grandeur of this historical fort and its beautiful gardens.
- Visit the Raj Ghat, the memorial of Mahatma Gandhi, to pay your respects and learn about his legacy.
- Conclude your trip with a farewell dinner at Dum Pukht, known for its slow-cooked Awadhi cuisine, set in an opulent environment. 

End of itinerary. Enjoy your luxurious experience in New Delhi!


In [9]:
# Cell 6: Export to Calendar (.ics file)
def export_to_ics(itinerary_text, start_date_str, destination_name):
    """
    Parse itinerary text and create a calendar file.
    
    Args:
        itinerary_text: The generated itinerary string
        start_date_str: Start date in YYYY-MM-DD format
        destination_name: The destination name
    
    Returns:
        Path to the created .ics file
    """
    
    # Parse start date
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    
    # Create calendar
    cal = Calendar()
    cal.add('prodid', '-//Travel Agent//EN')
    cal.add('version', '2.0')
    
    # Extract days from itinerary
    day_blocks = re.split(r'Day \d+', itinerary_text)
    
    day_num = 1
    for block in day_blocks[1:]:  # Skip first empty split
        # Extract day header (city/area)
        lines = block.strip().split('\n')
        if not lines:
            continue
        
        day_header = lines[0].strip(' -').strip()
        activities = [line.strip('- ').strip() for line in lines[1:] if line.strip().startswith('-')]
        
        # Calculate event date
        event_date = start_date + timedelta(days=day_num - 1)
        
        # Create event
        event = Event()
        event.add('summary', f'Day {day_num} - {day_header}')
        event.add('dtstart', event_date)
        event.add('dtend', event_date + timedelta(days=1))
        event.add('dtstamp', datetime.now())
        event.add('uid', f'{uuid.uuid4()}@travelagent.local')
        
        # Add activities as description
        description = '\n'.join(activities)
        event.add('description', description)
        
        cal.add_component(event)
        day_num += 1
    
    # Write to file
    filename = 'travel_itinerary.ics'
    with open(filename, 'wb') as f:
        f.write(cal.to_ical())
    
    return filename

# Generate the .ics file
ics_file = export_to_ics(itinerary, trip_start_date, destination)

print(f"\n✓ Calendar file created: {ics_file}")
print(f"  Import this file into Google Calendar, Apple Calendar, or Outlook")


✓ Calendar file created: travel_itinerary.ics
  Import this file into Google Calendar, Apple Calendar, or Outlook
